# 00 · Train everything on Colab (one click)

**Runtime → Change runtime type → A100 GPU** (or T4 on the free tier), then **Runtime → Run all**.

Getting the code (the notebook tries these in order; no setup needed if either of the first two applies):
1. **Drive zip:** put `RareCare.zip` in `MyDrive/rarecare/`.
2. **Public repo:** cloned directly from GitHub.
3. **Private repo:** add a Colab 🔑 Secret `GH_TOKEN` (read-only GitHub token).

*(Optional)* Put manually downloaded datasets (BUSI, SIPaKMeD, MMOTU, COCO subset) in `MyDrive/rarecare/raw/<name>/`.

Everything trained is saved to `MyDrive/rarecare/checkpoints/` and `MyDrive/rarecare/reports/`, so nothing is lost if the runtime disconnects.

| Step | What | T4 time (approx.) |
|---|---|---|
| 1 | StigmaSymp-W vignettes | < 1 min |
| 2 | BioBERT evidential concept classifier × 3 seeds | ~20 min |
| 3 | BioBERT symptom extractor | ~10 min |
| 4 | SapBERT lay → clinical alignment | ~5 min |
| 5 | ConvNeXt-Tiny vision (MedMNIST+ + CIFAR outliers + your Drive data) | ~40–90 min |
| 6 | Conformal calibration + held-out test check | ~5 min |
| 7 | Evaluation reports | ~5 min |

In [ ]:
import os, subprocess, sys
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/rarecare"
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True); os.makedirs(f"{DRIVE}/reports", exist_ok=True)

def get_code():
    if os.path.exists("/content/RareCare/.git"):
        subprocess.run(["git", "-C", "/content/RareCare", "pull", "-q"], check=False)
        return "existing (updated)"
    if os.path.exists("/content/RareCare"):
        return "existing"
    if os.path.exists(f"{DRIVE}/RareCare.zip"):
        subprocess.run(["unzip", "-q", f"{DRIVE}/RareCare.zip", "-d", "/content"], check=True)
        return "drive zip"
    url = "https://github.com/nush1729/RareCare.git"
    if subprocess.run(["git", "clone", "-q", url, "/content/RareCare"],
                      env={**os.environ, "GIT_TERMINAL_PROMPT": "0"}).returncode == 0:
        return "public GitHub"
    try:
        token = userdata.get("GH_TOKEN")
    except Exception:
        raise RuntimeError("Repo is private: put RareCare.zip in MyDrive/rarecare/ or add a GH_TOKEN secret.")
    subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/nush1729/RareCare.git", "/content/RareCare"], check=True)
    return "private GitHub"

print("code source:", get_code())
os.chdir("/content/RareCare")
# Colab already ships torch, torchvision, pandas, scikit-learn and matplotlib.
pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".",
                      "transformers", "timm", "datasets", "accelerate"],
                     capture_output=True, text=True)
if pip.returncode == 0:  # medmnist's CLI dep `fire` is sdist-only; the library itself needs none of it
    pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "medmnist"],
                         capture_output=True, text=True)
if pip.returncode != 0:
    print(pip.stderr[:6000])
    raise RuntimeError("pip install failed (see output above)")

# Persist outputs on Drive
for d in ("checkpoints", "reports"):
    if not os.path.islink(d):
        subprocess.run(["rm", "-rf", d]); os.symlink(f"{DRIVE}/{d}", d)
# Link any manually downloaded datasets from Drive
os.makedirs("data/raw", exist_ok=True)
if os.path.isdir(f"{DRIVE}/raw"):
    for name in os.listdir(f"{DRIVE}/raw"):
        if not os.path.exists(f"data/raw/{name}"):
            os.symlink(f"{DRIVE}/raw/{name}", f"data/raw/{name}")
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to GPU")

## 1 · StigmaSymp-W

In [ ]:
!python -m training.stigmasymp --n 6000 --seed 13 --out data/stigmasymp_w

## 2 · BioBERT evidential concept classifier (3 seeds)

In [ ]:
for seed in (13, 17, 23):
    subprocess.run([sys.executable, "-m", "training.train_text", "criteria", "--data", "data/stigmasymp_w",
                    "--out", f"checkpoints/biobert-criteria-s{seed}", "--epochs", "4", "--seed", str(seed)], check=True)
!cp -r checkpoints/biobert-criteria-s13 checkpoints/biobert-criteria

## 3 · BioBERT symptom extractor

In [ ]:
!python -m training.train_text extractor --data data/stigmasymp_w --out checkpoints/biobert-extractor --epochs 4 --fp16

## 4 · SapBERT alignment

In [ ]:
!python -m training.train_alignment --out checkpoints/sapbert-aligned --epochs 10

## 5 · Vision data + ConvNeXt-Tiny

In [ ]:
!python data/scripts/download_data.py --root data/raw --size 224 --router-size 64 --cap 1500 \
    --flags breastmnist retinamnist bloodmnist pneumoniamnist dermamnist
# Non-medical outliers: CIFAR-10 subset (outlier exposure)
import torchvision, pathlib
cifar = torchvision.datasets.CIFAR10("data/_cifar", train=True, download=True)
out = pathlib.Path("data/raw/outliers/cifar"); out.mkdir(parents=True, exist_ok=True)
for i in range(3000):
    cifar[i][0].resize((128, 128)).save(out / f"{i:05d}.png")
!python data/scripts/build_vision_manifest.py --root data/raw --out data/manifests/vision.csv

In [ ]:
!python -m training.train_vision --manifest data/manifests/vision.csv --out checkpoints/vision --epochs 15 --batch 64 --workers 2

## 6 · Conformal calibration (calibration half) and verification (test half)

In [ ]:
import numpy as np, pandas as pd, torch, json
from PIL import Image
from rarecare.vision.model import load_checkpoint, ORGAN_HEADS
from rarecare.vision.preprocess import to_model_input
from rarecare.uncertainty.conformal import PositiveClassConformal, empirical_recall
dev = "cuda" if torch.cuda.is_available() else "cpu"
model = load_checkpoint("checkpoints/vision/convnext_tiny.pt", dev)
m = pd.read_csv("data/manifests/vision.csv")
held = m[(m.split == "val") & (m.label >= 0)].sample(frac=1, random_state=13)
cal, test = held.iloc[: len(held) // 2], held.iloc[len(held) // 2 :]

def predict(df):
    rows = []
    for r in df.itertuples():
        x = torch.from_numpy(to_model_input(Image.open(r.path).convert("RGB")))[None].to(dev)
        with torch.no_grad():
            a = model(x)[f"evidence_{r.modality}"][0] + 1
        rows.append({"cancer": ORGAN_HEADS[r.modality], "p_suspicious": float(a[1] / a.sum()), "label": r.label, "group": r.source})
    return pd.DataFrame(rows)

cal_p, test_p = predict(cal), predict(test)
cal_p.to_csv("checkpoints/calib_preds.csv", index=False)
subprocess.run([sys.executable, "-m", "training.calibrate", "--preds", "checkpoints/calib_preds.csv",
                "--alpha", "0.05", "--out", "checkpoints/conformal.json"], check=True)
conf = json.load(open("checkpoints/conformal.json")); report = {}
for cancer, g in test_p.groupby("cancer"):
    c = conf[cancer]
    cp = PositiveClassConformal(alpha=c["alpha"], thresholds=c["thresholds"], n_calibration=c["n"])
    flags = np.array([cp.flag(p, s) for p, s in zip(g.p_suspicious, g.group, strict=True)])
    report[cancer] = {"test_recall": empirical_recall(flags, g.label.values), "flag_rate": float(flags.mean()),
                      "n_test": len(g), "n_pos": int((g.label == 1).sum())}
json.dump(report, open("reports/conformal_test.json", "w"), indent=2); report

## 7 · Evaluation reports

In [ ]:
!python -m eval.run_text_eval --data data/stigmasymp_w/val.jsonl --out reports/text_eval_rule_val.json > /dev/null
!python -m eval.run_text_eval --data data/stigmasymp_w/val.jsonl --config configs/trained_local.yaml --out reports/text_eval_trained_val.json > /dev/null
import json
for f in ("vision/history.json",):
    print(f, json.load(open(f"checkpoints/{f}"))[-1])
print(json.dumps(json.load(open("reports/text_eval_trained_val.json"))["by_register"], indent=1)[:1500])

## Done
Download `MyDrive/rarecare/checkpoints/` (or push to the HF Hub), then set `configs/trained.yaml` to those paths and the app switches from rule-baseline to neural mode.

⚠️ The validation numbers above come from **generated** vignettes. The reportable text result requires the **human-written test set** (see notebook 02).